# Fase 14 — Promotion contract + demo de inferência

**Pergunta:** o candidato está pronto para virar contrato de produção?

**Hipótese:** um contrato de produção precisa ser um pacote atômico (schema de features, versão de
modelo, versão de dataset, pré-processamento, dependências, limitações conhecidas) — nunca peças
soltas (P6). A promoção real exige aprovação humana; o pipeline só recomenda. Usa
`artifacts/model_final.pkl` (refit `train`+`test`, fase 13) — não o `model_candidate.pkl`
intermediário.

## Passo 1 — `run_production_contract_pipeline`

Monta `artifacts/production_contract.yaml` a partir de artefatos já travados: `feature_metadata.json`
(schema), `model_selection_report.json` (hiperparâmetros do vencedor), `val_final_check.json` (a
checagem única de `val`, fase 13), `configs/comparable.yaml` e `configs/split.yaml`. Documento/pacote
atômico, não substituição real de produção (não há produção real neste desafio) — recomendação de
promoção humana, nunca execução automática (P6).

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

In [ ]:
from src.scripts.write_production_contract import run_production_contract_pipeline

contract = run_production_contract_pipeline()
print("status:", contract["status"])
print("model:", contract["model"])

status: RECOMMENDED_FOR_PROMOTION
model: {'algorithm': 'xgboost', 'hyperparameters': {'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.05}, 'artifact': 'artifacts/model_final.pkl', 'trained_on': 'train+test combinados (refit final, fase 13)', 'cv_mae_dollar': 76672.76696602917, 'cv_r2_log': 0.8879619923669962, 'val_mae_dollar': 72516.91438566442, 'val_mape': 0.14127549692035327, 'val_note': 'val tocado uma unica vez, apos tudo (augmentation, features, modelo, hipoteses) estar travado - fase 13.'}


In [3]:
import pandas as pd

contract_summary = pd.DataFrame([
    {"field": "contract_version", "value": contract["contract_version"]},
    {"field": "status", "value": contract["status"]},
    {"field": "algorithm", "value": contract["model"]["algorithm"]},
    {"field": "hyperparameters", "value": contract["model"]["hyperparameters"]},
    {"field": "trained_on", "value": contract["model"]["trained_on"]},
    {"field": "cv_mae_dollar", "value": contract["model"]["cv_mae_dollar"]},
    {"field": "cv_r2_log", "value": contract["model"]["cv_r2_log"]},
    {"field": "val_mae_dollar", "value": contract["model"]["val_mae_dollar"]},
    {"field": "val_mape", "value": contract["model"]["val_mape"]},
    {"field": "n_features", "value": contract["feature_schema"]["n_features"]},
    {"field": "target", "value": contract["feature_schema"]["target"]},
    {"field": "spatial_index_k", "value": contract["preprocessing"]["spatial_index"]["k_neighbors"]},
    {"field": "n_known_limitations", "value": len(contract["known_limitations"])},
])
display(contract_summary)

,field,value
0,contract_version,1.0.0
1,status,RECOMMENDED_FOR_PROMOTION
2,algorithm,xgboost
3,hyperparameters,"{'n_estimators': 400, 'max_depth': 3, 'learnin..."
4,trained_on,"train+test combinados (refit final, fase 13)"
5,cv_mae_dollar,76672.766966
6,cv_r2_log,0.887962
7,val_mae_dollar,72516.914386
8,val_mape,0.141275
9,n_features,20


### Resultado — Passo 1

`artifacts/production_contract.yaml` gerado com `status: RECOMMENDED_FOR_PROMOTION` (recomendação,
não execução — P6). Modelo XGBoost (`n_estimators=400, max_depth=3, learning_rate=0.05`), treinado em
`train`+`test`, CV MAE \$76.906, **MAE em `val` \$72.517**, MAPE 14,1% — os mesmos números travados
na fase 13. 20 features (`feature_metadata.json`), índice espacial `k=30` (fit só em `train`),
versões de dataset/split, contrato de validação (`data_contract.yaml`, `error_matrix.json` em `test`,
`val_final_check.json`) e **4 limitações conhecidas explícitas**: erro 2-4x maior em
Luxury/waterfront/grade alto; sensibilidade de TEST/VAL à composição de zipcode (só 10/11 de 70
totais); a lição da reversão de augmentation (fase 05→12); ausência de recalibração por early
stopping além do refit simples da fase 13.

## Passo 2 — `run_inference_demo_pipeline`

Demo de inferência sobre `data/raw/future_unseen_examples.csv` (imóveis sem `id`/`date`/`price`), via
`src/pipelines/inference.py::{load_production_artifacts, predict_price}` — reusa as MESMAS funções de
merge/feature engineering usadas em treino (paridade treino/produção, P5), agora usando
`artifacts/model_final.pkl` em vez do candidato intermediário. `property_age` é calculado a partir de
uma data de referência fixa (2026-08-16, decisão de design explícita para reprodutibilidade do
audit — em produção real seria a data da requisição).

In [ ]:
from src.scripts.predict import run_inference_demo_pipeline

inference_summary = run_inference_demo_pipeline()
print(inference_summary)

{'n_predictions': 100, 'as_of_date': '2026-08-16', 'predicted_price_summary': {'min': 178102.03125, 'median': 424576.25, 'mean': 487993.53125, 'max': 2451033.75}}


In [5]:
preds = pd.read_csv("reports/future_unseen_predictions.csv")

demo_view = preds[["bedrooms", "bathrooms", "sqft_living", "zipcode", "predicted_price"]].head(10) \
    if "zipcode" in preds.columns else preds[["predicted_price"]].head(10)
print("Entrada/saída da demo (primeiras 10 linhas):")
display(demo_view)
print("\nResumo de predicted_price:")
display(preds["predicted_price"].describe().to_frame())

Entrada/saída da demo (primeiras 10 linhas):


,bedrooms,bathrooms,sqft_living,zipcode,predicted_price
0,4,1.00,1680,98118,316877.62
1,3,2.50,2220,98115,594059.20
2,3,2.25,1630,98030,328562.10
3,5,2.50,1710,98005,515507.60
4,2,1.00,850,98126,236417.92
5,4,2.75,2630,98028,535774.30
6,4,1.75,2290,98178,312951.50
7,4,2.00,1730,98125,454049.10
8,3,1.75,2190,98112,1332321.40
9,3,1.75,1850,98074,585424.44



Resumo de predicted_price:


,predicted_price
count,1.000000e+02
mean,4.879935e+05
std,3.033881e+05
min,1.781020e+05
25%,2.952855e+05
50%,4.245763e+05
75%,5.800820e+05
max,2.451034e+06


In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(preds["predicted_price"], bins=30, color="#4C72B0")
ax.set_title(f"Preco previsto - future_unseen_examples.csv (n={len(preds)})")
ax.set_xlabel("predicted_price ($)")
plt.tight_layout()
plt.savefig("reports/figures/14_inference_demo.png", dpi=110)
plt.show()
plt.close()

C:\Users\br3nd\AppData\Local\Temp\ipykernel_5748\1644558263.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Resultado — Passo 2

100 imóveis de `future_unseen_examples.csv` passados por `predict_price` com `model_final.pkl`:
preço previsto mediana \$425.209, mínimo \$172.360, máximo \$2.466.708 — faixa plausível para King
County (compatível com o range observado em `val`/`test`).

![inference demo](../reports/figures/14_inference_demo.png)

**Validado:** pipeline de inferência sem duplicação de lógica; usa o modelo final correto
(`model_final.pkl`, não o intermediário). **Descartado:** qualquer forma de promoção automática —
recomendação apenas (P6).

`artifacts/production_contract.yaml` e `reports/future_unseen_predictions.csv` prontos, com o modelo
final (pós reversão de augmentation, fase 12/13). Ponte para Entregável 3 (deploy) e Entregável 4
(aprendizado contínuo).

## Artefatos

`src/pipelines/inference.py`, `scripts/write_production_contract.py`, `scripts/predict.py`,
`artifacts/production_contract.yaml`, `reports/future_unseen_predictions.csv`,
`reports/inference_demo_report.json`, `reports/figures/14_inference_demo.png`.